In [1]:
# PyAST_test.py
from keymaerax_parser import KeYmaeraXParser, convert_keymaerax_ast_to_intermediate_pyast
from gym_generator import generate_gym_environment_code # Import the generator

# --- KeYmaera X Input ---
kx_input = """
Theorem "IJRR17/Theorem 1: Static safety"

Definitions
  Real ep;      /* time limit for control decisions */
  Real b;       /* minimum braking capability of the robot */
  Real A;       /* maximum acceleration -b <= a <= A */
  Real W;       /* maximum steering */

  Real stopDist(Real v) = v^2 / (2*b);
  Real accelComp(Real v) = ((A/b + 1) * (A/2 * ep^2 + ep*v));
  Real admissibleSeparation(Real v) = stopDist(v) + accelComp(v);

  Bool isWellformedDir(Real dx, Real dy) <-> dx^2 + dy^2 = 1;

  Bool bounds() <-> (
      A >= 0
    & b > 0
    & ep > 0
  );
  
  /* control */
  HP controlFunc ::= {
      /* brake on current curve or remain stopped */
      { a := -b; }
      ++
      { ?v = 0; a := 0; w := 0; }
      ++
      /* or choose a new safe curve */
      { a := A;
        w := *; ?-W<=w & w<=W;
        r := *;
        xo := *; yo := *;
        ?r!=0 & r*w = v;
        ? abs(x-xo) > admissibleSeparation(v)
        | abs(y-yo) > admissibleSeparation(v);
      }
    }; # Changed from '};;' to '};'

    HP stepFunction ::= { 
        x' = v * dx, y' = v * dy, v' = a,
        dx' = -w * dy, dy' = w * dx, w' = a/r,
        t' = 1 & t <= ep & v >= 0
      };
End.

ProgramVariables
  Real x;    /* robot position: x */
  Real y;    /* robot position: y */
  Real v;    /* robot translational velocity */
  Real a;    /* robot translational acceleration */
  Real dx;   /* robot orientation: x */
  Real dy;   /* robot orientation: y */
  Real w;    /* robot rotational velocity */
  Real r;    /* robot curve radius */
  Real xo;   /* position of closest obstacle on curve */
  Real yo;
  Real t;    /* time */
End.

Problem
assumptions(x, y, v, dx, dy, xo, yo) ->
  [
    {
      {
        controlFunc;
        t := 0;
      }

      /* dynamics */
      {
      stepFunction;@invariant(t>=0, isWellformedDir(dx, dy))
      }
    }*@invariant(loopinv(x, y, v, dx, dy, xo, yo))
  ](x - xo)^2 + (y - yo)^2 > 0
End.
"""

# --- Main Execution ---
if __name__ == "__main__":
    print("--- Parsing KeYmaera X Input into internal AST ---")
    parser = KeYmaeraXParser(kx_input)
    kx_ast = parser.parse()
    # Print the internal KeYmaera X AST for debugging/verification
    # print(kx_ast) # Commented out for brevity

    print("\n--- Converting KeYmaera X AST to Intermediate Python AST ---")
    py_ast_config = convert_keymaerax_ast_to_intermediate_pyast(kx_ast)
    print("Intermediate Python AST (GymEnvironmentConfig):\n", py_ast_config)

    print("\n--- Generating Gymnasium environment code ---")
    generated_code = generate_gym_environment_code(py_ast_config)

    output_filename = "generated_kx_env.py"
    with open(output_filename, "w") as f:
        f.write(generated_code)

    print(f"Generated Gymnasium environment code saved to {output_filename}")

    print("\n--- Instructions to use the generated environment ---")
    print(f"""
To use the generated environment, ensure `ast_config.py`, `keymaerax_parser.py`, and `gym_generator.py`
are in the same directory as this script. Then:

1.  Run `PyAST_test.py`. This will generate `generated_kx_env.py`.
2.  You can then import and use the generated environment in another Python script or interactive session:

```python
import gymnasium as gym
# The class name is derived from the Theorem name: "IJRR17/Theorem 1: Static safety"
# spaces, slashes, colons, etc., are replaced with underscores.
from generated_kx_env import IJRR17_Theorem_1_Static_safety 
import numpy as np

# Instantiate the environment
env = IJRR17_Theorem_1_Static_safety()

observation, info = env.reset()
print("Initial observation:", observation)
print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)

# The action space for this KeYmaera X example's controlFunc is a Dict.
# It represents a top-level choice between 3 branches, and some branches
# involve nondeterministic assignments (Box spaces).

# Action Space Example Structure (simplified):
# Dict(
#   'choice_ChoiceLevel_0': Discrete(3),  # 0, 1, or 2 for the three branches
#   'assign_w': Box(low=-inf, high=inf, shape=()), # For 'w := *' in branch 2
#   'assign_r': Box(low=-inf, high=inf, shape=()), # For 'r := *' in branch 2
#   'assign_xo': Box(low=-inf, high=inf, shape=()), # For 'xo := *' in branch 2
#   'assign_yo': Box(low=-inf, high=inf, shape=()), # For 'yo := *' in branch 2
# )

# When using a Dict action space, you must provide values for *all* keys,
# even if a chosen branch doesn't explicitly use a particular nondeterministic assignment.
# For unused Box spaces, you can provide a dummy value (e.g., 0.0).

# Let's assume some constant values for demonstration (these are set internally
# by the generated environment, but we need to know their effect on 'a'):
# b = some_positive_value (e.g., 1.0)
# A = some_positive_value (e.g., 1.0)
# W = some_positive_value (e.g., 1.0)

# Example 1: Choose the first branch: {{ a := -b; }} (index 0)
action_branch_0 = {{
    'choice_ChoiceLevel_0': 0, # Select the first branch
    'assign_w': 0.0,           # Dummy value for unused nondeterministic assignments
    'assign_r': 0.0,
    'assign_xo': 0.0,
    'assign_yo': 0.0,
}}

print("\\nTaking 5 steps with action_branch_0 (a := -b;):")
for i in range(5):
    observation, reward, terminated, truncated, info = env.step(action_branch_0)
    print(f"Step {{i+1}}: Obs={{observation}}, Reward={{reward:.2f}}, Terminated={{terminated}}, Truncated={{truncated}}")
    if terminated or truncated:
        print("Episode finished.")
        break
env.close()

# Example 2: Reset and choose the third branch: {{ a := A; w := *; ... }} (index 2)
# This branch requires specific values for w, r, xo, yo.
print("\\nResetting environment and taking 5 steps with action_branch_2 (a := A; w := *; etc.):")
observation, info = env.reset()

action_branch_2 = {{
    'choice_ChoiceLevel_0': 2, # Select the third branch
    'assign_w': 0.5,           # Example value for w (within -W to W, assuming W=1)
    'assign_r': 1.0,           # Example value for r (non-zero)
    'assign_xo': 10.0,         # Example value for xo
    'assign_yo': 0.0,          # Example value for yo
}}

for i in range(5):
    observation, reward, terminated, truncated, info = env.step(action_branch_2)
    print(f"Step {{i+1}}: Obs={{observation}}, Reward={{reward:.2f}}, Terminated={{terminated}}, Truncated={{truncated}}")
    if terminated or truncated:
        print("Episode finished.")
        break
env.close()
""")

--- Parsing KeYmaera X Input into internal AST ---
DEBUG: Source code cleanup complete.
DEBUG: KeYmaeraXParser initialized. Source code cleaned.
DEBUG: Starting parsing process.
DEBUG: Theorem name found: 'IJRR17/Theorem 1: Static safety'
DEBUG: Starting _parse_definitions.
DEBUG: Extracted block for 'Definitions'. Content length: 1007. Snippet: 'Real ep;                                            
  Real b;                                      ...'
DEBUG: Created DefinitionsBlock ASTNode.
DEBUG: Starting HP extraction.
DEBUG: Found HP header for 'controlFunc' at global index 506.
DEBUG: Extracted HP 'controlFunc' body (raw): '{
      { a := -b; }
      ++
      { ?v = 0; a := 0; w := 0; }
      ++
      { a := A;
        w :...'
DEBUG: HybridProgram 'controlFunc' calling _parse_hp_structure.
DEBUG: _parse_program_fragment called for: '{
      { a := -b; }
      ++
      { ?v = 0; a := 0; w := 0; }
      ++
      {...'
DEBUG: Stripped outer braces. Content: '{ a := -b; }
      ++
     

ValueError: StepFuncConfig not found in definitions.hybrid_programs

In [ ]:
import gymnasium as gym
# The class name is derived from the Theorem name: "IJRR17/Theorem 1: Static safety"
# spaces, slashes, colons, etc., are replaced with underscores.
from generated_kx_env import IJRR17_Theorem_1__Static_safety
import numpy as np

# Instantiate the environment
env = IJRR17_Theorem_1__Static_safety()

observation, info = env.reset()
print("Initial observation:", observation)
print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)

# The action space for this KeYmaera X example's controlFunc is a Dict.
# It represents a top-level choice between 3 branches, and some branches
# involve nondeterministic assignments (Box spaces).

# Action Space Example Structure (simplified):
# Dict(
#   'choice_ChoiceLevel_0': Discrete(3),  # 0, 1, or 2 for the three branches
#   'assign_w': Box(low=-inf, high=inf, shape=()), # For 'w := *' in branch 2
#   'assign_r': Box(low=-inf, high=inf, shape=()), # For 'r := *' in branch 2
#   'assign_xo': Box(low=-inf, high=inf, shape=()), # For 'xo := *' in branch 2
#   'assign_yo': Box(low=-inf, high=inf, shape=()), # For 'yo := *' in branch 2
# )

# When using a Dict action space, you must provide values for *all* keys,
# even if a chosen branch doesn't explicitly use a particular nondeterministic assignment.
# For unused Box spaces, you can provide a dummy value (e.g., 0.0).

# Let's assume some constant values for demonstration (these are set internally
# by the generated environment, but we need to know their effect on 'a'):
# b = some_positive_value (e.g., 1.0)
# A = some_positive_value (e.g., 1.0)
# W = some_positive_value (e.g., 1.0)

# Example 1: Choose the first branch: {{ a := -b; }} (index 0)
# FIX: Changed {{ to { and }} to }
action_branch_0 = {
    'choice_ChoiceLevel_0': 0, # Select the first branch
    'assign_w': 0.0,           # Dummy value for unused nondeterministic assignments
    'assign_r': 0.0,
    'assign_xo': 0.0,
    'assign_yo': 0.0,
}

print("\nTaking 5 steps with action_branch_0 (a := -b;):")
for i in range(5):
    observation, reward, terminated, truncated, info = env.step(action_branch_0)
    print(f"Step {i+1}: Obs={observation}, Reward={reward:.2f}, Terminated={terminated}, Truncated={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break
env.close()

# Example 2: Reset and choose the third branch: {{ a := A; w := *; ... }} (index 2)
# This branch requires specific values for w, r, xo, yo.
print("\nResetting environment and taking 5 steps with action_branch_2 (a := A; w := *; etc.):")
observation, info = env.reset()

# FIX: Changed {{ to { and }} to }
action_branch_2 = {
    'choice_ChoiceLevel_0': 2, # Select the third branch
    'assign_w': 0.5,           # Example value for w (within -W to W, assuming W=1)
    'assign_r': 1.0,           # Example value for r (non-zero)
    'assign_xo': 10.0,         # Example value for xo
    'assign_yo': 0.0,          # Example value for yo
}

for i in range(5):
    observation, reward, terminated, truncated, info = env.step(action_branch_2)
    print(f"Step {i+1}: Obs={observation}, Reward={reward:.2f}, Terminated={terminated}, Truncated={truncated}")

env.close()

DEBUG: _eval_expr called with expr_str='0.0'
DEBUG: full_context keys: dict_keys(['stopDist', 'accelComp', 'admissibleSeparation', 'isWellformedDir', 'bounds', 'assumptions', 'np'])
DEBUG: full_context['assumptions'] type: <class 'method'>, callable: True
DEBUG: sympy_locals_for_sympify keys: dict_keys(['stopDist', 'accelComp', 'admissibleSeparation', 'isWellformedDir', 'bounds', 'assumptions', 'np'])
DEBUG: sympy_locals_for_sympify['assumptions'] type: <class 'sympy.core.function.UndefinedFunction'>
DEBUG: After sympify, expr_str='0.0', sympy_expr=0.0, type=<class 'sympy.core.numbers.Float'>
DEBUG: Is sympy_expr a sympy.Function? False
DEBUG: Not an AppliedUndef. Proceeding to lambdify.
DEBUG: Lambdify path - lambdify_args: []
DEBUG: Lambdify path - lambdify_modules_dict keys: ['stopDist', 'accelComp', 'admissibleSeparation', 'isWellformedDir', 'bounds', 'assumptions']

DEBUG: _eval_expr called with expr_str='0.0'
DEBUG: full_context keys: dict_keys(['ep', 'stopDist', 'accelComp', 'ad

<lambdifygenerated-32>:2: RuntimeWarning: invalid value encountered in scalar divide
  return a/r
<lambdifygenerated-39>:2: RuntimeWarning: invalid value encountered in scalar divide
  return a/r


ValueError: Error evaluating expression '{expr_str}' with context '{full_context}': Sympify of expression 'could not parse '(t <== ep) and (v >== 0)'' failed, because of exception being raised:
SyntaxError: invalid syntax (<string>, line 1)